# 16장. 실험 기록과 최종 평가

모델을 만든 과정과 결과를 다른 사람이 다시 확인할 수 있도록 적는 방법, 그리고 남겨 둔 자료로 한 번만 하는 최종 평가.

**준비:** 서울 아파트 실거래가 CSV 네 개를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 끝까지 실행하는 데 5분쯤 걸린다.

## 16.1 실험을 기록하기: 자료, 설정, 환경, 난수

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.width", 200)

import os, re

def period(path):
    """파일 머리말에서 계약일자 범위를 읽는다. 실거래가 파일이 아니면 빈 리스트를 돌려준다."""
    with open(path, encoding="cp949", errors="ignore") as f:
        head = "".join(f.readline() for _ in range(15))
    return re.findall(r"계약일자 : (\d{4}-\d{2}-\d{2}) ~ (\d{4}-\d{2}-\d{2})", head)

files = sorted(f for f in os.listdir(".") if f.endswith(".csv") and period(f))
raw = pd.concat([pd.read_csv(f, encoding="cp949", skiprows=15, thousands=",",
                             dtype={"번지": str, "본번": str, "부번": str, "동": str}) for f in files],
                ignore_index=True)

v = raw[(raw["해제사유발생일"] == "-") & (raw["거래유형"] != "직거래")].copy()
v = v.rename(columns={"전용면적(㎡)": "면적"})
v["억"] = v["거래금액(만원)"] / 1e4
v["날짜"] = pd.to_datetime(v["계약년월"].astype(str) + v["계약일"].astype(str).str.zfill(2), format="%Y%m%d")
v["구"] = v["시군구"].str.split().str[1]
v["동네"] = v["시군구"].str.split().str[2]
v["동이름"] = v["구"] + " " + v["동네"]
v["단지"] = v["시군구"] + " " + v["번지"] + " " + v["단지명"]
v["연"] = v["날짜"].dt.year
v["연식"] = v["연"] - v["건축년도"]
v["달"] = (v["연"] - 2023) * 12 + v["날짜"].dt.month          # 2023년 1월이 1, 2025년 12월이 36
v = v.sort_values("날짜").reset_index(drop=True)
print(len(files), raw.shape, len(v))

In [ ]:
import time
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor

def off(actual_log, predicted_log):
    """오차율의 중앙값(%). 2장의 off와 같은 지표를 로그 가격에서 계산한다."""
    return round(float(np.median(np.abs(np.exp(predicted_log - actual_log) - 1)) * 100), 1)

def log_rmse(actual_log, predicted_log):
    return round(float(np.sqrt(((predicted_log - actual_log) ** 2).mean())), 4)

def table_of(df, ref, month=False):
    """면적, 층, 건축년도는 숫자 그대로 쓴다. 동과 단지는 ref에서 가나다순으로 매긴 번호로 바꾼다. 목록에 없는 이름은 -1이 된다."""
    dongs, complexes = pd.Index(sorted(ref["동이름"].unique())), pd.Index(sorted(ref["단지"].unique()))
    cols = [df["면적"], df["층"], df["건축년도"], dongs.get_indexer(df["동이름"]), complexes.get_indexer(df["단지"])]
    return np.column_stack(cols + ([df["달"]] if month else []))

from scipy import sparse
from sklearn.linear_model import Ridge

def design(df, ref):
    """10장의 특성 행렬에 계약한 달을 수치 특성 하나로 추가했다. 수치 특성 8개에 동과 단지의 가변수."""
    dongs, complexes = pd.Index(sorted(ref["동이름"].unique())), pd.Index(sorted(ref["단지"].unique()))
    num = np.column_stack([np.log(df["면적"]), df["층"], *((np.clip(df["연식"], 0, 59) // 10 == k).astype(float) for k in range(1, 6)), df["달"]])
    blocks = [sparse.csr_matrix(num)]
    for col, cats in [("동이름", dongs), ("단지", complexes)]:
        where_ = cats.get_indexer(df[col])
        known = where_ >= 0
        blocks.append(sparse.csr_matrix((np.ones(known.sum()), (np.where(known)[0], where_[known])), shape=(len(df), len(cats))))
    return sparse.hstack(blocks).tocsr()

def price(df):
    return np.log(df["억"].to_numpy())

def fit(made, kind, lam=0.01, rounds=1000, pick=2, trees=50, seed=0):
    """made로 모델을 추정해서, 거래 표를 받아 로그 가격을 예측하는 함수를 돌려준다. 12장과 13장의 세 모델."""
    if kind == "랜덤 포레스트":
        forest = RandomForestRegressor(trees, max_features=pick, n_jobs=-1, random_state=seed).fit(table_of(made, made, month=True), price(made))
        return lambda df: forest.predict(table_of(df, made, month=True))
    line = Ridge(alpha=lam, solver="sparse_cg", tol=1e-8, max_iter=20000).fit(design(made, made), price(made))
    if kind == "릿지":
        return lambda df: line.predict(design(df, made))
    mend = HistGradientBoostingRegressor(learning_rate=0.1, max_iter=rounds, early_stopping=False, random_state=seed)
    mend.fit(table_of(made, made, month=True), price(made) - line.predict(design(made, made)))       # 13.4절: 릿지의 잔차를 부스팅이 맡는다
    return lambda df: line.predict(design(df, made)) + mend.predict(table_of(df, made, month=True))

years = {y: v[v["연"] == y].reset_index(drop=True) for y in (2023, 2024, 2025, 2026)}
print({y: (len(d_), int(d_["달"].min()), int(d_["달"].max())) for y, d_ in years.items()})

In [ ]:
import hashlib, json, platform, sklearn

LOG = "experiments.jsonl"                            # 실험 한 번이 한 줄로 추가되는 기록 파일
open(LOG, "w").close()

def stamp(df):
    """자료의 요약. 줄 수, 기간, 그리고 내용이 한 칸이라도 바뀌면 달라지는 해시 값 여덟 글자."""
    body = pd.util.hash_pandas_object(df[["날짜", "단지", "면적", "층", "억"]], index=False).to_numpy().tobytes()
    return {"줄": len(df), "기간": f"{df['날짜'].min().date()}~{df['날짜'].max().date()}", "해시": hashlib.md5(body).hexdigest()[:8]}

def experiment(made, asked, kind, **setting):
    """모델을 추정하고 평가해서, 어떤 자료로 무엇을 어떻게 했는지와 결과를 기록 파일에 한 줄로 적는다."""
    began = time.time()
    model = fit(made, kind, **setting)
    said = model(asked)
    line = {"시각": time.strftime("%Y-%m-%d %H:%M:%S"), "모델": kind, "설정": setting,
            "추정 자료": stamp(made), "평가 자료": stamp(asked),
            "오차율": off(price(asked), said), "로그RMSE": log_rmse(price(asked), said), "걸린 초": round(time.time() - began, 1),
            "환경": {"python": platform.python_version(), "sklearn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__}}
    with open(LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(line, ensure_ascii=False) + "\n")
    return model

experiment(years[2023], years[2024], "릿지", lam=0.01)
with open(LOG, encoding="utf-8") as f:
    first = json.loads(f.readline())
for key, val in first.items():
    print(f"{key}: {val}")

In [ ]:
for pick in (2, 3):
    for seed in range(5):
        experiment(years[2023], years[2024], "랜덤 포레스트", pick=pick, trees=50, seed=seed)
for seed in (0, 1):
    experiment(years[2023], years[2024], "릿지 + 부스팅", lam=0.01, rounds=1000, seed=seed)

book = pd.read_json(LOG, lines=True)                 # 기록 파일을 표로 읽는다
book["설정 요약"] = book["설정"].map(lambda s: " ".join(f"{k}={val}" for k, val in s.items() if k != "seed"))
book["시드"] = book["설정"].map(lambda s: s.get("seed", ""))
print(book[["모델", "설정 요약", "시드", "오차율", "로그RMSE"]].to_string(index=False))

In [ ]:
by_setting = book.groupby(["모델", "설정 요약"])["로그RMSE"].agg(["count", "min", "mean", "max"]).round(4)
print(by_setting.to_string())
two = book[book["설정 요약"] == "pick=2 trees=50"]["로그RMSE"].to_numpy()
three = book[book["설정 요약"] == "pick=3 trees=50"]["로그RMSE"].to_numpy()
print(int((three[:, None] < two[None, :]).sum()), len(two) * len(three), round(float(two.std(ddof=1)), 4), round(float(three.mean() - two.mean()), 4))
err2 = book[book["설정 요약"] == "pick=2 trees=50"]["오차율"].to_numpy()
err3 = book[book["설정 요약"] == "pick=3 trees=50"]["오차율"].to_numpy()
print(err2.min(), err2.max(), err3.min(), err3.max(), int((err3[:, None] < err2[None, :]).sum()))

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba
from matplotlib.ticker import FuncFormatter

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)


def plain_ticks(axis):
    """로그 축의 눈금을 10의 거듭제곱 대신 보통 숫자로 적는다."""
    axis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}" if val >= 1 else f"{val:.7f}".rstrip("0")))

In [ ]:
# 그림 16-2
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4))
for ax, col, unit in [(axes[0], "로그RMSE", "2024년 로그 RMSE"), (axes[1], "오차율", "2024년 오차율의 중앙값(%)")]:
    for k, (name, color) in enumerate([("pick=2 trees=50", NAVY), ("pick=3 trees=50", BRICK)]):
        vals = book[book["설정 요약"] == name][col].to_numpy()
        ax.scatter(np.full(len(vals), k) + np.linspace(-0.12, 0.12, len(vals)), vals, s=34, color=color, zorder=3)
        ax.plot([k - 0.24, k + 0.24], [vals.mean()] * 2, color=color, lw=1.6)
        ax.text(k + 0.27, vals.mean(), f"평균 {vals.mean():.4f}" if col == "로그RMSE" else f"평균 {vals.mean():.2f}", fontsize=7.5, color=color, va="center")
    ax.set_xticks([0, 1], ["특성 2개씩 고려\n(max_features=2)", "특성 3개씩 고려\n(max_features=3)"], fontsize=8)
    ax.set_xlim(-0.5, 1.75)
    ax.set_ylabel(unit, fontsize=8)
    frame(ax, "점 하나가 시드 하나" if col == "로그RMSE" else "같은 실행을 오차율로 본 것")
fig.tight_layout()
fig.savefig("fig_16_2.png", dpi=300)
plt.show()

In [ ]:
entry = book[book["모델"] == "릿지 + 부스팅"].iloc[0]                           # 기록 파일에서 한 줄을 꺼낸다
print(entry["설정"], entry["추정 자료"], entry["로그RMSE"])

same = stamp(years[2023]) == entry["추정 자료"] and stamp(years[2024]) == entry["평가 자료"]
redo = fit(years[2023], entry["모델"], **entry["설정"])                         # 기록된 설정 그대로 다시 추정한다
print(same, log_rmse(price(years[2024]), redo(years[2024])), entry["로그RMSE"])

changed = years[2023].iloc[:-100]                                                # 거래 100건이 빠진 파일을 받았다고 하자
print(stamp(changed), stamp(changed) == entry["추정 자료"])
other = fit(changed, entry["모델"], **entry["설정"])
print(log_rmse(price(years[2024]), other(years[2024])), off(price(years[2024]), other(years[2024])), off(price(years[2024]), redo(years[2024])))

## 16.2 남겨 둔 자료로 한 번만 평가하기

In [ ]:
LEDGER = []                                          # 시험 자료를 쓴 일을 적는 목록

def open_test(name, model, test, why):
    said = model(test)
    LEDGER.append({"시각": time.strftime("%Y-%m-%d %H:%M"), "모델": name, "시험 자료": stamp(test), "이유": why, "오차율": off(price(test), said)})
    return said

until24, until25 = v[v["연"] <= 2024].reset_index(drop=True), v[v["연"] <= 2025].reset_index(drop=True)
earlier = fit(until24, "릿지 + 부스팅", lam=0.01, rounds=1000)                   # 13장의 모델: 2024년까지로 추정
said25 = earlier(years[2025])
final = fit(until25, "릿지 + 부스팅", lam=0.01, rounds=1000)                     # 고른 설정 그대로 2025년까지로 다시 추정한다
t26 = years[2026]
said26 = open_test("릿지 + 부스팅", final, t26, "모든 결정을 마친 뒤의 최종 평가")
print(pd.DataFrame(LEDGER)[["모델", "이유", "오차율"]].to_string(index=False))
print(LEDGER[0]["시험 자료"])
chain = pd.DataFrame([("2023년", "2024년", book[book["모델"] == "릿지 + 부스팅"]["오차율"].iloc[0], book[book["모델"] == "릿지 + 부스팅"]["로그RMSE"].iloc[0]),
                      ("2023~2024년", "2025년", off(price(years[2025]), said25), log_rmse(price(years[2025]), said25)),
                      ("2023~2025년", "2026년 1~10월", off(price(t26), said26), log_rmse(price(t26), said26))],
                     columns=["추정 자료", "평가 자료", "오차율", "로그 RMSE"])
print(chain.to_string(index=False))
print(round(float((said26 - price(t26)).mean() * 100), 1), round(float((said25 - price(years[2025])).mean() * 100), 1))

In [ ]:
gap = np.abs(np.exp(said26 - price(t26)) - 1) * 100                              # 거래마다의 오차율(%)
lean = (said26 - price(t26)) * 100                                               # 거래마다의 오차(예측 − 실제, 로그 가격 × 100)
rng = np.random.default_rng(0)
by_deal = [np.median(gap[rng.integers(0, len(gap), len(gap))]) for _ in range(1000)]                # 거래를 복원 추출
names, which = np.unique(t26["단지"].to_numpy(), return_inverse=True)
rows_of = [np.where(which == g)[0] for g in range(len(names))]
by_complex = [np.median(gap[np.concatenate([rows_of[g] for g in rng.integers(0, len(names), len(names))])]) for _ in range(300)]   # 단지를 복원 추출
print(np.quantile(by_deal, [0.025, 0.975]).round(2).tolist(), len(names), np.quantile(by_complex, [0.025, 0.975]).round(2).tolist())
per = np.bincount(which)
print(round(float(per.mean()), 1), int(np.median(per)), int(per.max()))

In [ ]:
# 그림 16-3
fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.2, 3.5), gridspec_kw={"width_ratios": [1.5, 1]})
labels = ["추정: 2023년\n평가: 2024년", "추정: 2023~2024년\n평가: 2025년", "추정: 2023~2025년\n평가: 2026년 1~10월"]
ax.bar(range(3), chain["오차율"], color=[GRAY, GRAY, BRICK], width=0.55)
for k, val in enumerate(chain["오차율"]):
    ax.text(k, val + 0.15, f"{val:.1f}%", fontsize=8, ha="center", color=NAVY)
ax.set_xticks(range(3), labels, fontsize=7.5)
ax.set_ylim(0, 8)
ax.set_ylabel("오차율의 중앙값(%)", fontsize=8)
frame(ax, "같은 설정을 세 시기에 평가한 결과")
ax.text(0.5, 7.3, "설정을 고를 때 본 자료", fontsize=7, color="#6b7280", ha="center")
ax.text(2, 7.3, "한 번만 쓴 시험 자료", fontsize=7, color=BRICK, ha="center")
point = float(np.median(gap))
for k, (name, draws, color) in enumerate([("거래를\n복원 추출", by_deal, NAVY), ("단지를\n복원 추출", by_complex, TEAL)]):
    lo_, hi_ = np.quantile(draws, [0.025, 0.975])
    bx.plot([k, k], [lo_, hi_], color=color, lw=2.2)
    bx.plot([k - 0.08, k + 0.08], [lo_, lo_], color=color, lw=2.2); bx.plot([k - 0.08, k + 0.08], [hi_, hi_], color=color, lw=2.2)
    bx.scatter([k], [point], s=40, color=color, zorder=3)
    bx.text(k + 0.14, hi_, f"{hi_:.2f}", fontsize=7.5, color=color, va="center"); bx.text(k + 0.14, lo_, f"{lo_:.2f}", fontsize=7.5, color=color, va="center")
bx.set_xticks([0, 1], ["거래를\n복원 추출", "단지를\n복원 추출"], fontsize=8)
bx.set_xlim(-0.5, 1.6)
bx.set_ylim(6.2, 7.0)
bx.set_ylabel("2026년 오차율의 95% 구간(%)", fontsize=8)
frame(bx, f"점은 관측값 {point:.2f}%")
fig.tight_layout()
fig.savefig("fig_16_3.png", dpi=300)
plt.show()

## 16.3 지표 하나로 요약하지 않기: 구간별 성능과 예측 구간

In [ ]:
def piece(mask):
    return int(mask.sum()), round(float(np.median(gap[mask])), 1), round(float(lean[mask].mean()), 1)

won, area = t26["억"].to_numpy(), t26["면적"].to_numpy()
new = ~t26["단지"].isin(until25["단지"]).to_numpy()
rows = [("전체", *piece(np.ones(len(t26), bool)))]
for name, lo, hi in [("5억 이하", 0, 5), ("5~10억", 5, 10), ("10~15억", 10, 15), ("15~20억", 15, 20), ("20~30억", 20, 30), ("30억 초과", 30, 9999)]:
    rows.append((name, *piece((won > lo) & (won <= hi))))
for name, lo, hi in [("40㎡ 이하", 0, 40), ("40~60㎡", 40, 60), ("60~85㎡", 60, 85), ("85~135㎡", 85, 135), ("135㎡ 초과", 135, 999)]:
    rows.append((name, *piece((area > lo) & (area <= hi))))
rows += [("처음 보는 단지", *piece(new)), ("이미 본 단지", *piece(~new))]
cut = pd.DataFrame(rows, columns=["구간", "거래 수", "오차율", "평균 오차"]).set_index("구간")
print(cut.to_string())
gu = pd.DataFrame({"구": t26["구"].to_numpy(), "오차율": gap, "평균 오차": lean}).groupby("구").agg(거래=("오차율", "size"), 오차율=("오차율", "median"), 평균오차=("평균 오차", "mean")).round(1).sort_values("오차율")
print(pd.concat([gu.head(3), gu.tail(3)]).to_string())

In [ ]:
# 그림 16-4
order = list(cut.index)[::-1]
colors = [GRAY if n == "전체" else BRICK if n == "처음 보는 단지" else NAVY for n in order]
fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.2, 4.6), sharey=True)
ax.barh(order, cut.loc[order, "오차율"], color=colors, height=0.62)
for yy, n in enumerate(order):
    ax.text(max(cut.loc[n, "오차율"], cut.loc["전체", "오차율"]) + 0.6, yy, f"{cut.loc[n, '오차율']:.1f}%  ({cut.loc[n, '거래 수']:,}건)", fontsize=7, va="center", color=NAVY)
ax.axvline(cut.loc["전체", "오차율"], color=GRAY, lw=0.9, ls="--", zorder=0)
ax.set_xlim(0, 39)
ax.set_xlabel("오차율의 중앙값(%)", fontsize=8)
frame(ax, "구간별 오차율. 점선은 전체", grid="x")
bx.barh(order, cut.loc[order, "평균 오차"], color=colors, height=0.62)
for yy, n in enumerate(order):
    val = cut.loc[n, "평균 오차"]
    bx.text(val + (0.5 if val >= 0 else -0.5), yy, f"{val:+.1f}", fontsize=7, va="center", ha="left" if val >= 0 else "right", color=NAVY)
bx.axvline(0, color="black", lw=0.7)
bx.set_xlim(-12, 21)
bx.set_xlabel("평균 오차(예측 - 실제, 로그 가격 × 100)", fontsize=8)
frame(bx, "0보다 크면 실제보다 높게 예측", grid="x")
for sep in (1.5, 6.5, 12.5):
    ax.axhline(sep, color="#c9d1da", lw=0.7); bx.axhline(sep, color="#c9d1da", lw=0.7)
ax.tick_params(axis="y", labelsize=8)
fig.tight_layout()
fig.savefig("fig_16_4.png", dpi=300)
plt.show()

In [ ]:
miss = price(years[2025]) - said25                                               # 2024년까지로 추정한 모델이 2025년에 남긴 오차(실제 − 예측)
low, high = np.quantile(miss, [0.1, 0.9])
print(round(float(low), 4), round(float(high), 4), round(float(np.exp(low)), 3), round(float(np.exp(high)), 3))

real = price(t26) - said26
inside = (real >= low) & (real <= high)                                          # 2026년의 실제 값이 그 구간 안에 있는가
print(round(float(inside.mean()), 3), round(float((real < low).mean()), 3), round(float((real > high).mean()), 3))
month = t26["달"].to_numpy()
cover = {int(m - 36): round(float(inside[month == m].mean()), 2) for m in np.unique(month)}
print(cover)

own = price(until25) - final(until25)                                            # 최종 모델이 훈련 자료에 남긴 오차로 구간을 만들면
lo_in, hi_in = np.quantile(own, [0.1, 0.9])
print(round(float(lo_in), 4), round(float(hi_in), 4), round(float(((real >= lo_in) & (real <= hi_in)).mean()), 3))
print(round(float(inside[new].mean()), 2), round(float(inside[~new].mean()), 2), round(float(inside[won > 30].mean()), 2), round(float(inside[(won > 5) & (won <= 10)].mean()), 2))
one = t26.iloc[[0]]
print(one[["구", "면적", "층", "건축년도", "억"]].to_string(index=False), np.exp(said26[0] + np.array([low, 0, high])).round(2).tolist())

In [ ]:
# 그림 16-5
fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.2, 3.6))
bins = np.linspace(-0.5, 0.7, 61)
ax.hist(miss, bins=bins, density=True, color=NAVY, alpha=0.55, label="2025년(구간을 정한 자료)")
ax.hist(real, bins=bins, density=True, histtype="step", color=BRICK, lw=1.4, label="2026년(시험 자료)")
for edge in (low, high):
    ax.axvline(edge, color="black", lw=0.9, ls="--")
ax.set_ylim(0, 7.6)
ax.text(low - 0.02, 6.9, f"10% 분위수\n{low:+.3f}", fontsize=7, ha="right", va="top", color=NAVY, bbox=BOX)
ax.text(high + 0.02, 4.4, f"90% 분위수\n{high:+.3f}", fontsize=7, ha="left", va="top", color=NAVY, bbox=BOX)
ax.set_xlabel("실제 - 예측(로그 가격)", fontsize=8)
ax.set_ylabel("밀도", fontsize=8)
ax.legend(frameon=False, fontsize=7, loc="upper right")
frame(ax, "오차의 분포와 80% 구간")
ms = sorted(cover)
bx.plot(ms, [cover[m] * 100 for m in ms], color=BRICK, lw=1.8, marker="o", ms=4)
for m in ms:
    bx.annotate(f"{cover[m] * 100:.0f}", (m, cover[m] * 100), xytext=(0, 6), textcoords="offset points", fontsize=7, color=BRICK, ha="center")
bx.axhline(80, color="black", lw=0.9, ls="--")
bx.text(10.3, 80.8, "목표 80%", fontsize=7, ha="right", color=NAVY)
bx.set_xticks(ms, [f"{m}월" for m in ms], fontsize=7.5)
bx.set_ylim(50, 95)
bx.set_xlabel("2026년의 계약 월", fontsize=8)
bx.set_ylabel("실제 가격이 구간 안에 들어간 비율(%)", fontsize=8)
frame(bx, "월별 포함 비율")
fig.tight_layout()
fig.savefig("fig_16_5.png", dpi=300)
plt.show()

## 16.4 시간이 지나면 나빠지는 모델: 다시 평가하고 다시 추정하기

In [ ]:
monthly = pd.DataFrame({"월": month - 36, "오차율": gap, "평균 오차": lean}).groupby("월").agg(거래=("오차율", "size"), 오차율=("오차율", "median"), 평균오차=("평균 오차", "mean")).round(2)
print(monthly.to_string())
level = pd.DataFrame({"달": v["달"], "억": v["억"]}).groupby("달")["억"].median()
print(round(float(level.loc[36]), 2), round(float(level.loc[40]), 2), round(float(level.loc[46]), 2), round(float(final(t26.iloc[[0]].assign(달=46))[0] - final(t26.iloc[[0]].assign(달=37))[0]), 4))

In [ ]:
fresh = []
for m in range(37, 47):
    again = fit(v[v["달"] < m].reset_index(drop=True), "릿지 + 부스팅", lam=0.01, rounds=1000)       # 그 달 전까지의 거래로 다시 추정한다
    part = t26[month == m]
    said = again(part)
    fresh.append((m - 36, off(price(part), said), round(float((said - price(part)).mean() * 100), 1)))
fresh = pd.DataFrame(fresh, columns=["월", "오차율", "평균 오차"]).set_index("월")
drift = pd.DataFrame({"고정한 모델": monthly["오차율"], "고정, 평균 오차": monthly["평균오차"], "매달 다시 추정": fresh["오차율"], "다시 추정, 평균 오차": fresh["평균 오차"]})
print(drift.to_string())

In [ ]:
# 그림 16-6
fig, (ax, bx) = plt.subplots(2, 1, figsize=(7.2, 5.4), sharex=True, gridspec_kw={"height_ratios": [1.5, 1]})
ms = drift.index.to_numpy()
ax.plot(ms, drift["고정한 모델"], color=BRICK, lw=1.9, marker="o", ms=4, label="2025년 12월까지로 추정하고 고정한 모델")
ax.plot(ms, drift["매달 다시 추정"], color=TEAL, lw=1.9, marker="s", ms=4, label="매달 그 전달까지의 거래로 다시 추정한 모델")
for m in ms:
    ax.annotate(f"{drift.loc[m, '고정한 모델']:.1f}", (m, drift.loc[m, "고정한 모델"]), xytext=(0, 6), textcoords="offset points", fontsize=7, color=BRICK, ha="center")
    ax.annotate(f"{drift.loc[m, '매달 다시 추정']:.1f}", (m, drift.loc[m, "매달 다시 추정"]), xytext=(0, -12), textcoords="offset points", fontsize=7, color=TEAL, ha="center")
ax.set_ylim(2.5, 10.5)
ax.set_ylabel("오차율의 중앙값(%)", fontsize=8)
ax.legend(frameon=False, fontsize=7.5, loc="upper left")
frame(ax, "2026년 월별 오차율")
w = 0.36
bx.bar(ms - w / 2, drift["고정, 평균 오차"], w, color=BRICK, label="고정한 모델")
bx.bar(ms + w / 2, drift["다시 추정, 평균 오차"], w, color=TEAL, label="매달 다시 추정한 모델")
bx.axhline(0, color="black", lw=0.7)
bx.set_xticks(ms, [f"{m}월" for m in ms], fontsize=8)
bx.set_ylabel("평균 오차\n(로그 가격 × 100)", fontsize=8)
bx.set_xlabel("2026년의 계약 월", fontsize=8)
bx.legend(frameon=False, fontsize=7.5, loc="lower left", ncol=2)
frame(bx, "0보다 작으면 실제보다 낮게 예측")
[a_.tick_params(labelbottom=True) for a_ in fig.axes]
fig.tight_layout()
fig.savefig("fig_16_6.png", dpi=300)
plt.show()

In [ ]:
card = {
    "모델": "릿지(λ = 0.01)의 예측 + 릿지 잔차에 대한 부스팅(학습률 0.1, 트리 1,000개)",
    "목표": "서울 아파트 매매 실거래가(억 원)의 로그",
    "추정 자료": stamp(until25),
    "제외한 거래": "계약 해제 거래, 직거래",
    "특성": "면적, 층, 건축년도(연식), 동, 단지, 계약한 달",
    "시험 자료": stamp(t26),
    "시험 자료를 쓴 횟수": len(LEDGER),
    "오차율의 중앙값(%)": f"{off(price(t26), said26)} (95% 구간, 단지 단위 부트스트랩: {np.quantile(by_complex, 0.025):.2f}~{np.quantile(by_complex, 0.975):.2f})",
    "로그 RMSE": log_rmse(price(t26), said26),
    "평균 오차(로그 가격 × 100)": round(float(lean.mean()), 1),
    "월별 오차율(1월 → 10월)": f"{monthly['오차율'].iloc[0]} → {monthly['오차율'].iloc[-1]}",
    "처음 보는 단지의 오차율": f"{cut.loc['처음 보는 단지', '오차율']} ({cut.loc['처음 보는 단지', '거래 수']}건)",
    "80% 예측 구간의 실제 포함 비율": f"{inside.mean() * 100:.0f}%",
    "환경": first["환경"],
}
for key, val in card.items():
    print(f"{key}: {val}")